# RF 베이스라인 기반 하이퍼파라미터 튜닝

`deal_model_paper_rf_baseline.ipynb`의 RF를 기준으로 고정하고, 동일한 30개 외부 분할에서 튜닝 효과를 비교한다.

- 입력 13개, 원본 448행, 행마다 Unknown 총 4개, 마스킹 10세트, 그룹 단위 Train/Test 80:20을 유지한다.
- **각 회차의 Train 안에서만** 그룹 5-Fold GridSearchCV를 수행한다. 외부 Test는 파라미터 선택에 사용하지 않는다.
- 선택 기준은 `neg_brier_score`다. 임계값은 0.5로 고정하고 Accuracy·AUC·Precision·Recall·F1·FP/FN도 평가한다.
- 총 64개 설정 × 내부 5-Fold × 외부 30회 = 9,600번의 검증 학습이다. 최적 설정 재학습과 기준 RF 재현 학습은 별도다.
- 기준 모델과 비교할 때 회차별 Test 10세트를 먼저 평균하고, 그다음 30회 평균을 계산한다.
- 저장할 회차는 베이스라인과 같은 1회차로 미리 고정한다. 점수가 가장 좋은 Test나 가장 자주 나온 파라미터를 보고 모델을 바꾸지 않는다.
- 기존 노트북·베이스라인 모델·백엔드 배포 모델은 변경하지 않는다. 이번 작업은 RF 튜닝이며 추가 앙상블이나 임계값 탐색은 하지 않는다.

논문에 없는 RF 파라미터를 우리가 탐색하는 단계다. 논문 원본 모델의 복원이 아니며,
이미 살펴본 데이터를 재사용하므로 완전히 새로운 외부 검증으로 해석하지 않는다.

실행 기록은 동일한 원본 CSV·베이스라인을 복사한 임시 작업 폴더에서 새 커널로 전체 재실행했다. 아래 상대 저장 경로는 그 임시 폴더 내부이며 기존 로컬·배포 산출물은 덮어쓰지 않았다. 실행 시간 등 메타데이터가 달라 새 파일의 해시는 기존 파일과 다를 수 있다.


## 1. 라이브러리


In [1]:
import hashlib
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedGroupKFold

## 2. 기존 전처리 재사용

전처리 노트북만 실행하고 전체 원본·마스킹 10세트를 가져온다.
전처리가 만드는 예전 7:3 분할은 사용하지 않는다. 원본 CSV 경로는 `SALESLUV_B2B_DATA_PATH` 설정을 따른다.


In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

ipython = get_ipython()
assert ipython is not None, "Jupyter 커널에서 실행해야 합니다."
# 전처리에서 출력하는 원본 영업 행은 이 노트북 출력에 다시 저장하지 않는다.
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
X_categorical = ipython.user_ns["X_categorical"]
X_all_masked_sets = ipython.user_ns["X_all_masked_sets"]
y = ipython.user_ns["y"]
input_group_ids = ipython.user_ns["input_group_ids"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]
UNKNOWN_COLUMNS_PER_ROW = ipython.user_ns["UNKNOWN_COLUMNS_PER_ROW"]

assert X_categorical.shape == (448, 13)
assert X_categorical.index.is_unique
assert X_categorical.index.equals(y.index)
assert y.index.equals(input_group_ids.index)
assert list(X_categorical.columns) == list(MODEL_FEATURE_NAMES)
assert set(y.unique()) == {0, 1}
assert len(X_all_masked_sets) == 10
assert UNKNOWN_COLUMNS_PER_ROW == 4
for masked_data in X_all_masked_sets.values():
    assert masked_data.index.equals(y.index)
    assert list(masked_data.columns) == list(MODEL_FEATURE_NAMES)
    assert masked_data.eq("Unknown").sum(axis=1).eq(UNKNOWN_COLUMNS_PER_ROW).all()
    assert (X_categorical.eq("Unknown") <= masked_data.eq("Unknown")).all().all()

# 각 마스킹 세트의 원본 행 순서와 정답 순서를 맞춘다.
X_all_raw = pd.concat(X_all_masked_sets, names=["mask_set", "original_row_id"])
y_all_masked = pd.concat(
    {set_name: y for set_name in X_all_masked_sets},
    names=["mask_set", "original_row_id"],
)
all_original_row_ids = X_all_raw.index.get_level_values("original_row_id")
assert X_all_raw.index.equals(y_all_masked.index)
assert X_all_raw.shape == (4480, 13)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 180)
print(f"원본 {len(y)}행, 입력 {len(MODEL_FEATURE_NAMES)}개")
print(f"동일 입력 {input_group_ids.nunique()}그룹, 마스킹 {len(X_all_masked_sets)}세트")

원본 448행, 입력 13개
동일 입력 198그룹, 마스킹 10세트


### 해석

13개 컬럼·원본 행 순서·마스킹 조건을 베이스라인과 동일하게 유지했다.
같은 입력의 중복 행과 마스킹 변형은 그룹으로 묶으며, 새 데이터처럼 취급하지 않는다.


## 3. 저장된 베이스라인과 분할 불러오기

이 프로젝트에서 직접 생성한 로컬 joblib만 읽는다. 다른 곳에서 받은 모델 파일을 임의로 대체하지 않는다.
기준 파일이 없으면 먼저 `deal_model_paper_rf_baseline.ipynb`를 실행한다.


In [3]:
artifact_dir = preprocessing_notebook.parents[1] / "pipeline" / "artifacts"
baseline_artifact_path = artifact_dir / "deal-paper-rf-baseline-v1.joblib"
assert baseline_artifact_path.exists(), "먼저 RF 베이스라인 노트북을 실행해야 합니다."
baseline_sha256 = hashlib.sha256(baseline_artifact_path.read_bytes()).hexdigest()
baseline_bundle = joblib.load(baseline_artifact_path)

assert baseline_bundle["model_version"] == "deal-paper-rf-baseline-v1"
assert baseline_bundle["source_sha256"] == SOURCE_SHA256
assert baseline_bundle["model_feature_names"] == list(MODEL_FEATURE_NAMES)
assert baseline_bundle["category_values"] == {
    column: list(values) for column, values in CATEGORY_VALUES.items()
}
assert baseline_bundle["evaluation"]["masking_set_count"] == len(X_all_masked_sets) == 10
assert baseline_bundle["evaluation"]["unknown_columns_per_row"] == UNKNOWN_COLUMNS_PER_ROW
for package, saved_version in baseline_bundle["versions"].items():
    assert version(package) == saved_version, f"{package} 버전이 기준 실험과 다릅니다."

# clone은 학습된 나무를 버리고 설정만 복사한다. Test를 본 모델을 이어 학습하지 않는다.
rf_template = clone(baseline_bundle["model"])
evaluation_splits = baseline_bundle["evaluation"]["splits"]
saved_baseline_results = baseline_bundle["evaluation"]["repeat_results"]
saved_baseline_masks = baseline_bundle["evaluation"]["mask_results"].set_index(
    ["repeat", "mask_set"]
)
CLASSIFICATION_THRESHOLD = baseline_bundle["classification_threshold"]
REFERENCE_REPEAT = baseline_bundle["reference_repeat"]
INNER_CV_FOLDS = 5
INNER_CV_RANDOM_STATE = 1
SCORING = "neg_brier_score"
assert len(evaluation_splits) == 30
assert CLASSIFICATION_THRESHOLD == 0.5
assert REFERENCE_REPEAT == 1
assert rf_template.named_steps["classifier"].get_params() == baseline_bundle["rf_params"]
display(baseline_bundle["evaluation"]["summary"].round(6))

,30회_평균,반복간_표준편차
accuracy,0.695847,0.039228
auc,0.740949,0.045089
brier,0.209706,0.019724
precision,0.695281,0.062083
recall,0.740104,0.064721
f1,0.714223,0.050065
fp,15.593333,4.740066
fn,12.543333,4.434480
tn,27.440000,3.498433
tp,36.256667,11.406824


### 해석

기준 점수와 원본 Train/Test 위치를 파일에서 직접 가져온다.
RF도 저장 모델의 설정을 복사하므로 다른 기본값으로 바뀌지 않는다.
아래에서 기본 RF를 다시 학습해 마스킹 세트별 점수까지 기준 기록과 일치하는지 확인한다.


## 4. GridSearch 탐색 범위


In [4]:
param = {
    "classifier__n_estimators": [100, 300],
    "classifier__max_depth": [None, 4, 8, 12],
    "classifier__min_samples_leaf": [1, 5, 10, 20],
    "classifier__max_features": ["sqrt", 0.5],
}
parameter_combinations = list(ParameterGrid(param))
baseline_params = {name: rf_template.get_params()[name] for name in param}
assert len(parameter_combinations) == 64
assert baseline_params in parameter_combinations
display(pd.Series(param, name="탐색값").to_frame())
print(
    f"{len(parameter_combinations)}개 설정 × {INNER_CV_FOLDS}-Fold × "
    f"{len(evaluation_splits)}회 = "
    f"{len(parameter_combinations) * INNER_CV_FOLDS * len(evaluation_splits):,}번 검증 학습"
)

,탐색값
classifier__n_estimators,"[100, 300]"
classifier__max_depth,"[None, 4, 8, 12]"
classifier__min_samples_leaf,"[1, 5, 10, 20]"
classifier__max_features,"[sqrt, 0.5]"


64개 설정 × 5-Fold × 30회 = 9,600번 검증 학습


### 해석

나무 수는 평균화 정도, 깊이와 최소 잎 크기는 모델의 복잡도, 분기 피처 수는 나무 사이의 무작위성을 조절한다.
최소 잎 크기는 마스킹 변형을 포함한 행 기준이지 원본 거래 건수가 아니다.

기본 RF 설정도 64개 후보에 포함했다. 따라서 같은 내부 CV에서 최적 점수가 기본보다 나쁘지 않은지 확인할 수 있다.
그렇다고 외부 Test 점수까지 반드시 개선되는 것은 아니다. 탐색 범위는 Test 평가 전에 고정한다.


## 5. 각 Train 내부 튜닝과 동일 Test 평가

바깥 Train/Test 분할과 안쪽 파라미터 선택을 분리한다.
안쪽 5-Fold에서도 동일 원본 ID와 입력 그룹의 마스킹 변형을 같은 쪽에 둔다.

CPU 병렬화는 GridSearch에만 적용하고 RF 내부는 1개 작업으로 유지한다.


In [5]:
def calculate_metrics(y_true, probability):
    """0.5 임계값으로 확률·분류 지표를 함께 계산한다."""
    assert np.isfinite(probability).all()
    assert ((probability >= 0) & (probability <= 1)).all()
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, prediction),
        "auc": roc_auc_score(y_true, probability),
        "brier": brier_score_loss(y_true, probability),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "tp": int(tp),
        "fpr": float(fp / (fp + tn)),
    }


mask_result_rows = []
repeat_rows = []
search_result_frames = []
selection_rows = []
tuning_started = perf_counter()

for repeat, (train_positions, test_positions) in enumerate(evaluation_splits, start=1):
    repeat_started = perf_counter()
    train_row_ids = y.index[train_positions]
    test_row_ids = y.index[test_positions]
    assert set(train_row_ids).isdisjoint(test_row_ids)
    assert set(train_row_ids) | set(test_row_ids) == set(y.index)
    assert set(input_group_ids.iloc[train_positions]).isdisjoint(
        input_group_ids.iloc[test_positions]
    )
    assert set(y.iloc[train_positions].unique()) == {0, 1}
    assert set(y.iloc[test_positions].unique()) == {0, 1}

    train_mask = all_original_row_ids.isin(train_row_ids)
    X_repeat_train = X_all_raw.loc[train_mask]
    y_repeat_train = y_all_masked.loc[train_mask]
    repeat_original_ids = X_repeat_train.index.get_level_values("original_row_id")
    repeat_groups = input_group_ids.loc[repeat_original_ids].to_numpy()
    assert X_repeat_train.index.equals(y_repeat_train.index)
    assert len(X_repeat_train) == len(train_positions) * 10
    assert pd.Series(repeat_original_ids).value_counts().eq(10).all()

    inner_cv = StratifiedGroupKFold(
        n_splits=INNER_CV_FOLDS,
        shuffle=True,
        random_state=INNER_CV_RANDOM_STATE,
    )
    inner_splits = list(inner_cv.split(X_repeat_train, y_repeat_train, groups=repeat_groups))
    for inner_train, inner_valid in inner_splits:
        assert set(repeat_groups[inner_train]).isdisjoint(repeat_groups[inner_valid])
        assert set(repeat_original_ids[inner_train]).isdisjoint(repeat_original_ids[inner_valid])
        assert set(y_repeat_train.iloc[inner_train].unique()) == {0, 1}
        assert set(y_repeat_train.iloc[inner_valid].unique()) == {0, 1}
    coverage = np.bincount(
        np.concatenate([valid for _, valid in inner_splits]), minlength=len(X_repeat_train)
    )
    assert (coverage == 1).all()

    search_rf = GridSearchCV(
        rf_template,
        param,
        cv=inner_splits,
        scoring=SCORING,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        error_score="raise",
        verbose=1,
    )
    search_rf.fit(X_repeat_train, y_repeat_train)
    baseline_index = search_rf.cv_results_["params"].index(baseline_params)
    baseline_cv_score = search_rf.cv_results_["mean_test_score"][baseline_index]
    assert search_rf.best_score_ >= baseline_cv_score - 1e-12
    assert len(search_rf.cv_results_["params"]) == len(parameter_combinations)

    search_results = pd.DataFrame(
        {
            "repeat": repeat,
            "rank": search_rf.cv_results_["rank_test_score"],
            "params": search_rf.cv_results_["params"],
            "cv_brier": -search_rf.cv_results_["mean_test_score"],
            "cv_brier_std": search_rf.cv_results_["std_test_score"],
            "train_brier": -search_rf.cv_results_["mean_train_score"],
            "mean_fit_seconds": search_rf.cv_results_["mean_fit_time"],
        }
    )
    search_result_frames.append(search_results)
    selection_rows.append(
        {
            "repeat": repeat,
            "baseline_cv_brier": -baseline_cv_score,
            "tuned_cv_brier": -search_rf.best_score_,
            "best_params": search_rf.best_params_,
        }
    )

    # 기준 RF를 같은 Train에서 재학습해 저장된 기존 결과와 일치하는지 검증한다.
    model_rf_base = clone(rf_template).fit(X_repeat_train, y_repeat_train)
    model_rf_tuned = search_rf.best_estimator_
    for model_name, estimator in [
        ("RandomForest_base", model_rf_base),
        ("RandomForest_tuned", model_rf_tuned),
    ]:
        won_index = list(estimator.classes_).index(1)
        mask_metrics = []
        for set_name, masked_data in X_all_masked_sets.items():
            X_repeat_test = masked_data.iloc[test_positions]
            y_repeat_test = y.iloc[test_positions]
            assert X_repeat_test.index.equals(y_repeat_test.index)
            probability = estimator.predict_proba(X_repeat_test)[:, won_index]
            metrics = calculate_metrics(y_repeat_test, probability)
            if model_name == "RandomForest_base":
                np.testing.assert_allclose(
                    list(metrics.values()),
                    saved_baseline_masks.loc[(repeat, set_name), list(metrics)].to_numpy(),
                    rtol=1e-10,
                    atol=1e-12,
                )
            mask_metrics.append(metrics)
            mask_result_rows.append(
                {"repeat": repeat, "model": model_name, "mask_set": set_name, **metrics}
            )
        mean_metrics = pd.DataFrame(mask_metrics).mean().to_dict()
        repeat_rows.append(
            {
                "repeat": repeat,
                "model": model_name,
                "train_rows": len(train_positions),
                "test_rows": len(test_positions),
                **mean_metrics,
            }
        )
        if model_name == "RandomForest_tuned":
            tuned_metrics = mean_metrics

    if repeat == REFERENCE_REPEAT:
        reference_tuned_model = model_rf_tuned
        reference_best_params = search_rf.best_params_.copy()
        reference_inner_splits = inner_splits

    elapsed = perf_counter() - repeat_started
    base_metrics = saved_baseline_results.loc[repeat]
    print(
        f"{repeat:02d}/30 완료 ({elapsed:.1f}초) | "
        f"Brier {base_metrics['brier']:.4f} → {tuned_metrics['brier']:.4f}, "
        f"Accuracy {base_metrics['accuracy']:.4f} → {tuned_metrics['accuracy']:.4f}, "
        f"AUC {base_metrics['auc']:.4f} → {tuned_metrics['auc']:.4f}",
        flush=True,
    )

tuning_seconds = perf_counter() - tuning_started
mask_results = pd.DataFrame(mask_result_rows)
repeat_results = pd.DataFrame(repeat_rows)
search_results = pd.concat(search_result_frames, ignore_index=True)
selection_results = pd.DataFrame(selection_rows).set_index("repeat")
metric_names = list(tuned_metrics)
assert len(mask_results) == 600
assert len(repeat_results) == 60
assert len(search_results) == 30 * 64
assert mask_results.groupby(["model", "repeat"]).size().eq(10).all()
assert repeat_results.groupby("model")["repeat"].nunique().eq(30).all()
assert selection_results["tuned_cv_brier"].le(selection_results["baseline_cv_brier"] + 1e-12).all()
np.testing.assert_allclose(
    mask_results.groupby(["model", "repeat"])[metric_names].mean(),
    repeat_results.set_index(["model", "repeat"])[metric_names].sort_index(),
)
np.testing.assert_allclose(
    repeat_results[["fp", "fn", "tn", "tp"]].sum(axis=1),
    repeat_results["test_rows"],
)
assert np.isfinite(repeat_results[metric_names].to_numpy()).all()

Fitting 5 folds for each of 64 candidates, totalling 320 fits


01/30 완료 (10.2초) | Brier 0.2314 → 0.2085, Accuracy 0.6846 → 0.7209, AUC 0.7043 → 0.7262


Fitting 5 folds for each of 64 candidates, totalling 320 fits


02/30 완료 (8.7초) | Brier 0.2180 → 0.1901, Accuracy 0.6598 → 0.7207, AUC 0.7282 → 0.7699


Fitting 5 folds for each of 64 candidates, totalling 320 fits


...

28/30 완료 (8.2초) | Brier 0.2393 → 0.2089, Accuracy 0.6126 → 0.6916, AUC 0.6602 → 0.7039


Fitting 5 folds for each of 64 candidates, totalling 320 fits


29/30 완료 (8.8초) | Brier 0.2248 → 0.1982, Accuracy 0.6679 → 0.7060, AUC 0.7146 → 0.7798


Fitting 5 folds for each of 64 candidates, totalling 320 fits


30/30 완료 (8.8초) | Brier 0.1942 → 0.1855, Accuracy 0.7186 → 0.7477, AUC 0.7862 → 0.7949


### 해석

파라미터는 안쪽 검증 Brier로만 선택하고, 바깥 Test에서는 고정된 모델을 평가했다.
기본 RF의 Test 300세트 점수를 모두 재현해 다른 분할이나 마스킹 결과를 비교하지 않았는지 확인했다.

각 회차에서 best params가 달라질 수 있다. 아래 30회 평균은 하나의 고정 파라미터가 아니라
**Train에서 튜닝하는 절차 전체**를 평가한 결과다. 내부 CV 점수는 선택에 사용했으므로 외부 Test 성능으로 대신 읽지 않는다.


## 6. 기본 RF와 튜닝 RF의 30회 비교


In [6]:
comparison = repeat_results.groupby("model", sort=False)[metric_names].mean()
comparison_std = repeat_results.groupby("model", sort=False)[metric_names].std()
baseline_by_repeat = repeat_results.query("model == 'RandomForest_base'").set_index("repeat")[
    metric_names
]
tuned_by_repeat = repeat_results.query("model == 'RandomForest_tuned'").set_index("repeat")[
    metric_names
]
paired_delta = tuned_by_repeat - baseline_by_repeat
delta_summary = pd.DataFrame(
    {
        "기본_RF_평균": comparison.loc["RandomForest_base"],
        "튜닝_RF_평균": comparison.loc["RandomForest_tuned"],
        "튜닝_마이너스_기본": paired_delta.mean(),
    }
)
# 같은 값의 뺄셈에서 생기는 미세한 부동소수점 오차를 개선으로 세지 않는다.
improved_counts = pd.Series(
    {
        "Brier 감소": int(paired_delta["brier"].lt(-1e-12).sum()),
        "Accuracy 증가": int(paired_delta["accuracy"].gt(1e-12).sum()),
        "AUC 증가": int(paired_delta["auc"].gt(1e-12).sum()),
        "FP 감소": int(paired_delta["fp"].lt(-1e-12).sum()),
        "FPR 감소": int(paired_delta["fpr"].lt(-1e-12).sum()),
    },
    name="개선된 회차 수 / 30",
)
display(delta_summary.round(6))
display(comparison_std.rename_axis("반복간 표준편차").round(6))
display(improved_counts.to_frame())
display(paired_delta[["accuracy", "auc", "brier", "fp", "fn", "fpr"]].round(6))
print(f"전체 튜닝·재현·평가 시간: {tuning_seconds:.1f}초")

,기본_RF_평균,튜닝_RF_평균,튜닝_마이너스_기본
accuracy,0.695847,0.733833,0.037985
auc,0.740949,0.775560,0.034611
brier,0.209706,0.190198,-0.019508
precision,0.695281,0.706008,0.010727
recall,0.740104,0.835410,0.095306
f1,0.714223,0.763253,0.049030
fp,15.593333,16.766667,1.173333
fn,12.543333,7.636667,-4.906667
tn,27.440000,26.266667,-1.173333
tp,36.256667,41.163333,4.906667


,accuracy,auc,brier,precision,recall,f1,fp,fn,tn,tp,fpr
반복간 표준편차,,,,,,,,,,,
RandomForest_base,0.039228,0.045089,0.019724,0.062083,0.064721,0.050065,4.740066,4.434480,3.498433,11.406824,0.077161
RandomForest_tuned,0.044744,0.047202,0.017526,0.065069,0.063904,0.053539,5.366328,2.486582,3.926157,13.712403,0.092293


,개선된 회차 수 / 30
Brier 감소,30
Accuracy 증가,28
AUC 증가,29
FP 감소,6
FPR 감소,6


,accuracy,auc,brier,fp,fn,fpr
repeat,,,,,,
1,0.036264,0.021890,-0.022898,0.5,-3.8,0.011905
2,0.060920,0.041693,-0.027966,2.3,-7.6,0.058974
3,0.028750,0.037492,-0.015652,-0.1,-2.2,-0.002222
4,0.041667,0.050052,-0.023322,1.9,-6.4,0.038776
5,-0.003488,0.051599,-0.023935,4.6,-4.3,0.112195
6,0.060484,0.049584,-0.012731,2.8,-10.3,0.065116
7,0.029231,0.029832,-0.014247,1.1,-3.0,0.028205
8,0.043038,0.030618,-0.017745,0.6,-4.0,0.018750
9,0.005085,0.030238,-0.014909,1.4,-1.7,0.040000


전체 튜닝·재현·평가 시간: 264.5초


### 해석

차이는 튜닝 점수에서 기본 점수를 뺀 값이다. Accuracy·AUC는 양수, Brier·FP·FPR은 음수면 해당 지표가 개선됐다.
Brier가 좋아져도 FP가 늘었거나 Accuracy가 떨어진 경우를 함께 확인한다.

FP는 실제 Lost를 Won으로 표시해 주의해야 할 거래를 놓치는 경우다.
FPR은 실제 Lost 중 이런 실수를 한 비율이다. Test 행 수가 반복마다 달라 건수와 비율을 같이 본다.
30회 반복은 같은 데이터가 겹치므로 개선 회차 수를 독립 표본의 유의성 검정으로 해석하지 않는다.


## 7. 회차별 최적 파라미터와 기준 회차 탐색 결과


In [7]:
display(selection_results.round(6))
reference_search_results = search_results.loc[
    search_results["repeat"].eq(REFERENCE_REPEAT)
].sort_values("rank", kind="stable")
display(reference_search_results.drop(columns="repeat").round(6))
print(f"저장할 {REFERENCE_REPEAT}회차 최적 파라미터: {reference_best_params}")

,baseline_cv_brier,tuned_cv_brier,best_params
repeat,,,
1,0.200449,0.184536,"{'classifier__max_depth': 12, 'classifier__max..."
2,0.202741,0.187852,"{'classifier__max_depth': 12, 'classifier__max..."
3,0.214362,0.193392,"{'classifier__max_depth': 8, 'classifier__max_..."
4,0.204951,0.185527,"{'classifier__max_depth': 8, 'classifier__max_..."
5,0.193699,0.179456,"{'classifier__max_depth': 8, 'classifier__max_..."
6,0.217181,0.198246,"{'classifier__max_depth': 8, 'classifier__max_..."
7,0.212841,0.192258,"{'classifier__max_depth': 8, 'classifier__max_..."
8,0.212612,0.192214,"{'classifier__max_depth': 12, 'classifier__max..."
9,0.204860,0.182765,"{'classifier__max_depth': 8, 'classifier__max_..."


,rank,params,cv_brier,cv_brier_std,train_brier,mean_fit_seconds
51,1,"{'classifier__max_depth': 12, 'classifier__max...",0.184536,0.021088,0.137464,0.368515
50,2,"{'classifier__max_depth': 12, 'classifier__max...",0.184561,0.020836,0.137720,0.130237
3,3,"{'classifier__max_depth': None, 'classifier__m...",0.185336,0.022384,0.129637,0.394479
35,4,"{'classifier__max_depth': 8, 'classifier__max_...",0.185571,0.019638,0.152910,0.375578
49,5,"{'classifier__max_depth': 12, 'classifier__max...",0.185594,0.019350,0.106493,0.376696
...,...,...,...,...,...,...
22,60,"{'classifier__max_depth': 4, 'classifier__max_...",0.196134,0.014947,0.182337,0.098797
1,61,"{'classifier__max_depth': None, 'classifier__m...",0.198762,0.025831,0.035480,0.520922
0,62,"{'classifier__max_depth': None, 'classifier__m...",0.200449,0.026118,0.035946,0.188736
9,63,"{'classifier__max_depth': None, 'classifier__m...",0.207099,0.025551,0.035167,0.813851


저장할 1회차 최적 파라미터: {'classifier__max_depth': 12, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 300}


### 해석

1회차 표는 학습된 행의 Brier와 학습에 쓰지 않은 내부 검증 행의 Brier를 나란히 보여준다.
학습 점수만 좋고 검증 점수가 나쁘면 과적합을 의심할 수 있다.

저장 모델의 설정은 1회차 Train의 GridSearch에서 정한 값이다.
다른 회차의 Test 점수나 파라미터 등장 빈도로 저장 모델을 다시 고르지 않는다.
나머지 회차의 64개 전체 탐색 결과도 아래 산출물에 보관한다.


## 8. 튜닝 모델 저장과 재로드 검증


In [8]:
known_row = {
    column: next(value for value in CATEGORY_VALUES[column] if value != "Unknown")
    for column in MODEL_FEATURE_NAMES
}
masked_row = {**known_row, **dict.fromkeys(MODEL_FEATURE_NAMES[:4], "Unknown")}
self_check_X = pd.DataFrame(
    [known_row, masked_row, dict.fromkeys(MODEL_FEATURE_NAMES, "Unknown")],
    columns=list(MODEL_FEATURE_NAMES),
)
expected_probability = reference_tuned_model.predict_proba(self_check_X)
assert np.isfinite(expected_probability).all()
np.testing.assert_allclose(expected_probability.sum(axis=1), 1.0)

artifact_path = artifact_dir / "deal-paper-rf-tuned-v1.joblib"
bundle = {
    "schema_version": 1,
    "model_version": "deal-paper-rf-tuned-v1",
    "model": reference_tuned_model,
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "source_sha256": SOURCE_SHA256,
    "baseline_model_version": baseline_bundle["model_version"],
    "baseline_artifact_sha256": baseline_sha256,
    "baseline_rf_params": baseline_bundle["rf_params"],
    "training_scope": "reference_split_train_only",
    "reference_repeat": REFERENCE_REPEAT,
    "training_original_rows": baseline_bundle["training_original_rows"],
    "training_masked_rows": baseline_bundle["training_masked_rows"],
    "reference_train_positions": baseline_bundle["reference_train_positions"],
    "reference_test_positions": baseline_bundle["reference_test_positions"],
    "reference_test_metrics": tuned_by_repeat.loc[REFERENCE_REPEAT].to_dict(),
    "best_params": reference_best_params,
    "rf_params": reference_tuned_model.named_steps["classifier"].get_params(),
    "tuning": {
        "search": "GridSearchCV",
        "scoring": SCORING,
        "param_grid": param,
        "inner_cv": "StratifiedGroupKFold",
        "inner_cv_n_splits": INNER_CV_FOLDS,
        "inner_cv_random_state": INNER_CV_RANDOM_STATE,
        "reference_inner_splits": reference_inner_splits,
        "selection_results": selection_results,
        "search_results": search_results,
    },
    "evaluation": {
        "splitter": baseline_bundle["evaluation"]["splitter"],
        "split_random_state": baseline_bundle["evaluation"]["split_random_state"],
        "test_group_fraction": baseline_bundle["evaluation"]["test_group_fraction"],
        "repeat_count": len(evaluation_splits),
        "masking_set_count": len(X_all_masked_sets),
        "unknown_columns_per_row": UNKNOWN_COLUMNS_PER_ROW,
        "splits": evaluation_splits,
        "repeat_results": repeat_results,
        "mask_results": mask_results,
        "comparison_mean": comparison,
        "comparison_std": comparison_std,
        "paired_delta": paired_delta,
        "seconds": tuning_seconds,
    },
    "versions": {name: version(name) for name in ("scikit-learn", "numpy", "pandas", "joblib")},
}
joblib.dump(bundle, artifact_path)
restored = joblib.load(artifact_path)
np.testing.assert_allclose(
    expected_probability,
    restored["model"].predict_proba(self_check_X),
    rtol=1e-12,
    atol=1e-12,
)
assert set(restored["model"].classes_) == {0, 1}
assert restored["model"].named_steps["onehot"].transform(self_check_X).shape == (3, 39)
assert len(restored["model_feature_names"]) == 13
for name, value in restored["best_params"].items():
    assert restored["model"].get_params()[name] == value
assert hashlib.sha256(baseline_artifact_path.read_bytes()).hexdigest() == baseline_sha256
print(f"튜닝 모델 저장: backend/pipeline/artifacts/{artifact_path.name}")
print(f"파일 크기: {artifact_path.stat().st_size / 1024**2:.3f} MiB")
print("기준 결과 재현·내부/외부 그룹 분리·전체 반복 평가·저장 후 재로드: 통과")

튜닝 모델 저장: backend/pipeline/artifacts/deal-paper-rf-tuned-v1.joblib
파일 크기: 7.761 MiB
기준 결과 재현·내부/외부 그룹 분리·전체 반복 평가·저장 후 재로드: 통과


### 해석

파일에는 1회차의 튜닝 RF와 원핫 인코더, 입력 계약, 파라미터와 전체 비교 결과가 들어 있다.
30회 평균은 튜닝 절차의 평가이며 저장 파일 하나의 성적이 아니다. 파일의 `reference_test_metrics`가 1회차 결과다.

기존 베이스라인 파일이 바뀌지 않았는지 해시도 확인했다.
Test를 포함해 전체 재학습하거나 AWS·백엔드 모델을 교체하지 않았다. 모델 파일은 Git에 올리지 않는다.
